# Chapter 09: Mechanism Design & Tragedy of the Commons for Shared Infrastructure
### *Incentive Compatibility, Moral Hazard of Automated Retries, and Pigouvian Flakiness Quotas*

**[Course Home](../README.md) | [Python Lab](../python/09_mechanism_design_and_tragedy_of_the_commons.ipynb) | [Prev: Ch 08](08_quickest_change_detection_cusum_and_shiryaev_roberts.ipynb) | [Next: Appendix 10: Rosetta Stone](10_appendix_optimization_and_control_rosetta_stone.ipynb)**

---

## 1. Motivation: The Tragedy of the Infrastructure Commons

In modern software organizations, continuous integration (CI/CD) pipelines, compute clusters, merge queues, and automated testbeds are managed as **shared common-pool resources**:
* Hundreds of engineers across multiple product teams push pull requests (PRs) to a unified repository.
* A shared fleet of cloud worker nodes executes build targets and test suites.
* A global reliability budget $B_{\text{suite}}$ or acceptable failure tolerance is maintained across the organization.

When engineering leadership attempts to protect developer productivity from flaky tests or transient failures by introducing **automated retries** (e.g., "automatically retry any failed test up to 3 times before failing the build"), they inadvertently trigger a classic microeconomic market failure: **Moral Hazard** and the **Tragedy of the Commons**.

### The Moral Hazard Mechanism
1. **Zero Private Marginal Cost**:
   When an individual team authors an intermittent, flaky, or non-hermetic test that fails $20\%$ of the time, the automated retry policy absorbs the defect:
   $$\mathbb{P}(\text{All 3 Retries Fail}) = (0.20)^3 = 0.008 \quad (0.8\%)$$
   To the authoring team, the test passes $99.2\%$ of the time! Their PR merges, their feature ships, and their sprint goals are met with zero localized penalty.
2. **Diffuse Social Marginal Cost**:
   However, every time that test runs, it consumes an average of:
   $$\mathbb{E}[\text{Executions}] = 1 + 0.20 + (0.20)^2 = 1.24 \text{ runs}$$
   Multiply this $24\%$ compute inflation across $5{,}000$ tests in a monorepo, and the worker cluster experiences severe queue congestion. Every peer team in the company experiences delayed merge times, stale PR re-runs, and starved worker pools.

```
 +-------------------------------------------------------------------------+
 |                      GLOBAL SHARED RUNNER POOL                          |
 |                                                                         |
 |  Team A: Low Quality PR (20% flake)  ---> Burns 3x retries             |
 |  Team B: Low Quality PR (15% flake)  ---> Burns 3x retries             |
 |  Team C: High Quality PR (0% flake)  ---> STUCK IN QUEUE FOR 4 HOURS!   |
 +-------------------------------------------------------------------------+
       RESULT: Unregulated Nash Equilibrium = Complete Cluster Collapse
```

Without economic mechanism design, the rational game-theoretic strategy for every team is to **free-ride** on the global retry budget, ceasing all investment in test hermeticity and flake remediation until the shared infrastructure grinds to a halt.

---

## 2. Game-Theoretic Formulation of Infrastructure Sharing

### 2.1 The $K$-Team Non-Cooperative Game
Let $K$ distinct engineering teams share a fixed compute pool with total worker capacity $C$.
Each team $k \in \{1, \dots, K\}$ chooses an **effort level** $e_k \in [0, 1]$ dedicated to test engineering, hermetic mocking, and stability hardening:
* Higher effort $e_k$ increases test pass reliability: $p_k = e_k$.
* Lower effort yields higher failure probability: $1 - e_k$.

Team $k$ incurs a private cost of effort $c(e_k)$, where $c(\cdot)$ is strictly increasing, convex, and satisfies $c'(e) \to \infty$ as $e \to 1$:

$$
c(e_k) = \frac{\theta_k}{1 - e_k}
$$

where $\theta_k > 0$ represents the difficulty or technical debt of team $k$'s codebase.

### 2.2 The Retry Expansion Factor
Let $R \ge 0$ denote the maximum number of automated retries permitted by the CI system.
If a team's test has failure probability $q_k = 1 - e_k$, the number of executions consumed before either passing or exhausting all retries follows a truncated geometric series:

$$
N_k(e_k) = 1 + q_k + q_k^2 + \dots + q_k^R = \frac{1 - q_k^{R+1}}{1 - q_k} = \frac{1 - (1 - e_k)^{R+1}}{e_k}
$$

If team $k$ submits jobs at nominal arrival rate $\lambda_k$, their effective load on the shared cluster is:

$$
L_k(e_k) = \lambda_k N_k(e_k)
$$

Total cluster arrival rate:

$$
L(\mathbf{e}) = \sum_{k=1}^K L_k(e_k) = \sum_{k=1}^K \lambda_k \frac{1 - (1 - e_k)^{R+1}}{e_k}
$$

### 2.3 Congestion Delay and Team Payoff Functions
Modeling the runner cluster as an $M/M/C$ or high-traffic queue, the average turnaround latency delay $W(L)$ experienced by all teams grows non-linearly with total load:

$$
W(L) = \frac{1}{C - L(\mathbf{e})} \quad \text{for } L(\mathbf{e}) < C
$$

Each team derives gross business value $V_k$ from shipping features, but suffers private effort cost $c(e_k)$ and queue waiting delay $\delta_k W(L)$:

$$
U_k(e_k; \mathbf{e}_{-k}) = V_k - c(e_k) - \delta_k W(L(\mathbf{e}))
$$

---

## 3. Nash Equilibrium vs. Social Welfare Optimum

### 3.1 The Unregulated Nash Equilibrium
In a decentralized environment without governance, each team unilaterally chooses $e_k$ to maximize its own private utility:

$$
\max_{e_k \in [0, 1]} U_k(e_k; \mathbf{e}_{-k}) = V_k - \frac{\theta_k}{1 - e_k} - \frac{\delta_k}{C - \sum_{j=1}^K L_j(e_j)}
$$

Setting the first-order condition $\frac{\partial U_k}{\partial e_k} = 0$:

$$
c'(e_k^{\text{Nash}}) = -\delta_k W'(L) \cdot \frac{\partial L_k}{\partial e_k}
$$

$$
\frac{\theta_k}{(1 - e_k^{\text{Nash}})^2} = \frac{\delta_k}{(C - L)^2} \cdot \left| \frac{\partial L_k}{\partial e_k} \right|
$$

### 3.2 The Social Welfare Optimum (Centralized Planner)
A centralized infrastructure architect seeks to maximize total aggregate utility across all $K$ teams:

$$
\max_{\mathbf{e}} \mathcal{W}(\mathbf{e}) = \sum_{k=1}^K U_k(e_k; \mathbf{e}_{-k}) = \sum_{k=1}^K \left[ V_k - c(e_k) \right] - \left(\sum_{k=1}^K \delta_k\right) W(L(\mathbf{e}))
$$

Setting the social first-order condition $\frac{\partial \mathcal{W}}{\partial e_k} = 0$:

$$
c'(e_k^{\text{Social}}) = -\left( \sum_{j=1}^K \delta_j \right) W'(L) \cdot \frac{\partial L_k}{\partial e_k}
$$

### 3.3 The Externality Gap
Compare the two first-order conditions:
* At the Nash equilibrium, team $k$ internalizes only its own delay sensitivity $\delta_k$.
* At the Social optimum, the marginal effort is weighted by the **entire community's aggregate delay sensitivity** $\sum_{j=1}^K \delta_j$.

Because $\sum_{j=1}^K \delta_j \approx K \cdot \bar{\delta} \gg \delta_k$, we have:

$$
c'(e_k^{\text{Social}}) \approx K \cdot c'(e_k^{\text{Nash}}) \implies e_k^{\text{Nash}} \ll e_k^{\text{Social}}
$$

Every team under-invests in stability by a factor proportional to $K$! As the number of teams sharing the infrastructure grows, private investment in stability collapses toward zero, and the cluster enters permanent saturation.

---

## 4. Mechanism Design Countermeasures

To realign private incentives with social optimality, the infrastructure organization can implement two mechanism design interventions:

### 4.1 Pigouvian Congestion Pricing
Arthur Pigou (1920) proved that negative externalities can be perfectly neutralized by imposing a marginal tax equal to the social damage inflicted on others.
The **Pigouvian infrastructure tax** $\tau_k$ charged to team $k$ (in internal compute budget or chargeback tokens) is:

$$
\tau_k = \left( \sum_{j \ne k} \delta_j \right) W'(L) \cdot \left[ L_k(e_k) - \lambda_k \right]
$$

Under this tax, team $k$'s effective utility becomes:

$$
U_k^{\text{tax}}(e_k) = U_k(e_k) - \tau_k
$$

Taking the derivative shows that team $k$'s private first-order condition becomes mathematically identical to the Social Planner's objective!

### 4.2 Incentive-Compatible Per-Team Flake Quotas & SLA Demotion
When monetary chargebacks are infeasible inside an engineering organization, the Vickrey-Clarke-Groves (VCG) principle is enforced via **strict per-team reliability quotas**:

```
                    +------------------------------------+
                    |  Team Flake Budget B_k = B_tot / K |
                    +------------------------------------+
                                      |
                +---------------------+---------------------+
                |                                           |
         Runs <= B_k                                   Runs > B_k
   (Compliant Team)                              (Quota Exceeded)
        |                                                   |
  v-----v--------------v                      v-------------v--------------v
  | Tier 1: Fast Lane  |                      | Tier 2: Slow Pre-emptible  |
  | Global Retries ON  |                      | Retries STRICTLY DISABLED  |
  | SLA: < 5 min wait  |                      | SLA: Best Effort Queue     |
  +--------------------+                      +----------------------------+
```

1. **The Quota Allocation**: Each team receives an invariant monthly flake budget $B_k = B_{\text{global}} / K$.
2. **SLA Demotion Rule**:
   * As long as team $k$'s test runs stay within $B_k$, their PRs enjoy high-priority worker execution and automated retries.
   * The moment team $k$'s flaky test runs exceed $B_k$, automated retries are **immediately disabled** for that team's PRs, and their jobs are demoted to a low-priority, pre-emptible queue.
3. **Incentive Alignment**: Under SLA demotion, a team cannot shift its flakiness onto other teams. A flaky test directly blocks *only the authoring team's own developers*. The moral hazard vanishes!

---

## 5. Concrete Step-by-Step Worked Examples with Real Numbers

### 5.1 Worked Example: 5 Teams Sharing an 80-Worker CI Cluster
Consider an engineering organization with $K = 5$ teams sharing a CI runner pool with capacity $C = 80$ concurrent workers:
* Nominal job submission rate: $\lambda_k = 12$ jobs/hour for each team $k \in \{1, \dots, 5\}$.
* Total baseline arrival rate: $\Lambda_0 = 5 \times 12 = 60$ jobs/hour.
* Effort cost function: $c(e_k) = \frac{2.0}{1 - e_k}$ ($e_k \in [0.5, 0.99]$).
* Delay sensitivity: $\delta_k = 1.0$ for all teams ($\sum_{j=1}^5 \delta_j = 5.0$).
* Cluster queue delay: $W(L) = \frac{20}{80 - L}$ minutes.
* Automated retry policy: $R = 2$ retries (maximum 3 runs per job).

#### Scenario A: Unregulated Nash Equilibrium (Global Retries Enabled)
Each team optimizes privately:
$$\max_{e_k} \left[ V_k - \frac{2.0}{1 - e_k} - \frac{20}{80 - L} \right]$$
Solving the coupled equilibrium numerically yields:
$$e_k^{\text{Nash}} \approx 0.72 \quad (72\% \text{ test reliability, } 28\% \text{ flake})$$

Expected runs per job:
$$N(0.72) = 1 + 0.28 + (0.28)^2 = 1 + 0.28 + 0.0784 = 1.3584 \text{ runs/job}$$

Total cluster load:
$$L_{\text{Nash}} = 5 \times 12 \times 1.3584 = 60 \times 1.3584 = 81.5 \text{ jobs/hour} > C = 80!$$

**Catastrophic Result**:
Because $L_{\text{Nash}} > 80$, the queue is non-stationary! Queue length and merge delays explode to infinity. The CI cluster crashes, merge queues freeze, and engineering productivity drops to zero.

---

#### Scenario B: Social Optimum (Centralized Planner)
The social planner minimizes total cost across all 5 teams:
$$\min_{e} \left[ 5 \times \frac{2.0}{1 - e} + 5 \times \frac{20}{80 - 60 N(e)} \right]$$

Testing $e = 0.95$ ($95\%$ test reliability, $5\%$ flake):
$$N(0.95) = 1 + 0.05 + (0.05)^2 = 1.0525 \text{ runs/job}$$
$$L_{\text{Social}} = 60 \times 1.0525 = 63.15 \text{ jobs/hour}$$

Queue delay:
$$W(L_{\text{Social}}) = \frac{20}{80 - 63.15} = \frac{20}{16.85} \approx 1.19 \text{ minutes}$$

Total cost per team:
$$\text{Effort Cost} = \frac{2.0}{1 - 0.95} = \frac{2.0}{0.05} = 40.0$$
$$\text{Delay Cost} = 1.19$$
$$\text{Total Cost} = 41.19$$

---

#### Scenario C: Pigouvian Per-Team Flake Quota Mechanism
The platform team imposes a per-team flake quota:
* Each team is allocated a maximum of $12 \times 1.06 = 12.72$ runner hours per hour ($e_k \ge 0.94$).
* If a team's tests exceed this quota, automated retries are disabled for that team ($R = 0$).

Faced with the penalty of having retries disabled (which would cause $28\%$ of their PRs to fail immediately and block their sprint), team management rationally allocates developer hours to test stability, choosing $e_k = 0.94$.
* Total load: $L = 60 \times 1.0636 = 63.82$ jobs/hour.
* Queue delay: $W = \frac{20}{80 - 63.82} = 1.24$ minutes.
* Cluster stability: Fully preserved in stationary equilibrium!

---

## 6. Engineering Takeaways and Practical Architecture

| Architecture | Retry Governance | Equilibrium Outcome | Queue Latency |
| :--- | :--- | :--- | :--- |
| **Unregulated Commons** | Unconditional global retries ($R=3$) | Free-riding tragedy; cluster saturation ($L \ge C$) | $\infty$ (merge queue freeze) |
| **Complete Ban on Retries** | Zero retries ($R=0$) | High developer friction; false PR rejections | Low latency, but high developer toil |
| **Per-Team Flake Quotas** | Retries allowed within per-team budget $B_k$ | **Incentive-Compatible Nash Equilibrium** | **$1.2$ minutes (Optimal)** |
| **SLA Demotion** | Retries disabled only for non-compliant teams | Localized accountability; zero cross-team contagion | Protected fast-lane for compliant teams |

---
**[Course Home](../README.md) | [Python Lab](../python/09_mechanism_design_and_tragedy_of_the_commons.ipynb) | [Prev: Ch 08](08_quickest_change_detection_cusum_and_shiryaev_roberts.ipynb) | [Next: Appendix 10: Rosetta Stone](10_appendix_optimization_and_control_rosetta_stone.ipynb)**
